In [ ]:

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import parmed
import MDAnalysis as mda


In [ ]:

df = pd.read_csv('../gxg_comps/3_res/total_conts_df.csv',
                # header=[0,1,2],low_memory=False
                )


In [ ]:

df = pd.read_csv('../gxg_comps/9_res/total_conts_df.csv',
                # header=[0,1,2],low_memory=False
                )


In [ ]:

df = pd.read_csv('../gxg_comps/9_res/equal/2_close/plf_files/all_conts.csv',
                # header=[0,1,2],low_memory=False
                )


In [ ]:
df = pd.read_csv('../gxg_comps/3_res/aromatic/plf_files/ALA_plf.csv',
                 header=[0,1,2],low_memory=False
                )

In [ ]:
df

In [ ]:
pwd

In [ ]:

systems = ['equal','charged','aromatic','polar','nonpolar']

df_3res = pd.read_csv('data/2_close_cluster_water_9res.csv')
df_9res = pd.read_csv('data/sasa_persidue_9res.csv')


In [ ]:
### Contacts formed in cluster 

In [ ]:
import parmed
from sklearn.cluster import DBSCAN
from MDAnalysis.analysis import contacts
from MDAnalysis.analysis import distances
from collections import Counter
import nglview as nv
import MDAnalysis as mda
import numpy as np
from MDAnalysis.transformations import wrap, center_in_box
from scipy.spatial.distance import pdist,squareform
import seaborn as sns
import pytim
import warnings
import pandas as pd
import matplotlib.pyplot as plt
import os
from MDAnalysis.analysis import contacts, distances
from MDAnalysis import transformations as trans
from scipy import sparse

In [ ]:

### For RNA G and A, it is 13
### For RNA U and C, it is 11.5

def clustering(u, frame,eps=13):

        ### Function that is nested in all of the analysis methods in this file ###

        ca_sel = u.select_atoms("name C1' or name CA")

        cas = [atom.index for atom in ca_sel.atoms]

        to_cluster = [atom for i, atom in enumerate(u.trajectory[frame.frame].positions)if i in cas]

        n_ca = len(ca_sel)

        self_distances = distances.self_distance_array(ca_sel.positions,
                                                       box=frame.dimensions)

        sq_dist_arr = np.zeros((n_ca, n_ca))
        triu = np.triu_indices_from(sq_dist_arr, k=1)

        sq_dist_arr[triu] = self_distances
        sq_dist_arr.T[triu] = self_distances
  
        model = DBSCAN(eps=eps, min_samples=1, metric='precomputed')
    
        labels = model.fit_predict(sq_dist_arr)
        b = Counter(labels)

        clustered_list = [atom for j, atom in enumerate(to_cluster)
                          for i, item in enumerate(labels)
                          if item == b.most_common()[0][0]
                          and j == i]

        centroid = np.mean(clustered_list, axis=0)

        ind_list = [cas[index] for index, j in enumerate(to_cluster)
                    for i in clustered_list
                    if j[0]==i[0] and j[1]==i[1] and j[2]==i[2]]

        index_list = [x+1 for x in ind_list]

        res_list = [u.atoms[i].resindex for i in index_list]

        return centroid, to_cluster, clustered_list, res_list
    
    
    
def surface_comp2(u, start=4000, stop=-1, step=10000):
    frame_num = 0
    inner_layers_all = []
    outer_layers_all = []

    for frame in u.trajectory[start:stop:step]:
        
        centroid, to_cluster, clustered_list, res_list = clustering(u, frame, eps=9)

        # --- Build selection for clustered core residues ---
        selection = 'resindex '
        for item in res_list:
            selection += str(item) + ' or resindex '
        selection = selection[0:-13]

        sel = u.select_atoms(selection)

        # --- Write clustered residues to temp PDB and reload ---
        with mda.Writer('spatial.pdb') as pdb:
            pdb.write(sel)
        univ = mda.Universe('spatial.pdb')

        # --- Run GITIM surface analysis ---
        g = univ.select_atoms('name CA')
        interface = pytim.GITIM(
            univ,
            group=g,
            alpha=4.5,
            max_layers=3,
            molecular=False,
            centered='middle'
        )

        # --- Extract layer 1 (outer) and layer 2 (inner) via interface object ---
        layer1_atoms = interface.layers[0]   # outer surface
        layer2_atoms = interface.layers[1]   # mid layer
        layer3_atoms = interface.layers[2]   # inner layer

        outer_resids = np.unique(layer1_atoms.resids)
        mid_resids = np.unique(layer2_atoms.resids)
        inner_resids = np.unique(layer3_atoms.resids)
        
        
        return inner_resids ,mid_resids,outer_resids,res_list
    

def surface_comp3(u, start=4000, stop=-1, step=10000):
    
    frame_num = 0
    inner_layers_all = []
    outer_layers_all = []

    for frame in u.trajectory[start:stop:step]:
        centroid, to_cluster, clustered_list, res_list = clustering(u, frame, eps=9)

        # --- Build selection for clustered core residues ---
        selection = 'resindex '
        for item in res_list:
            selection += str(item) + ' or resindex '
        selection = selection[0:-13]

        sel = u.select_atoms(selection)

        # --- Write clustered residues to temp PDB and reload ---
        with mda.Writer('spatial.pdb') as pdb:
            pdb.write(sel)
        univ = mda.Universe('spatial.pdb')

        # --- Run GITIM surface analysis ---
        g = univ.select_atoms('name CA')
        interface = pytim.GITIM(
            univ,
            group=g,
            alpha=4.2,
            max_layers=2,
            molecular=False,
            centered='middle'
        )

        # --- Extract layer 1 (outer) and layer 2 (inner) via interface object ---
        layer1_atoms = interface.layers[0]   # outer surface
        layer2_atoms = interface.layers[1]   # mid layer
        #layer3_atoms = interface.layers[2]   # inner layer

        outer_resids = np.unique(layer1_atoms.resids)
        inner_resids = np.unique(layer2_atoms.resids)
        
        return inner_resids,outer_resids,res_list


def surface_comp_new(u):
    
    inner_layers_all = []
    outer_layers_all = []
    frame = u.trajectory[0]
    
    centroid, to_cluster, clustered_list, res_list = clustering(u,frame, eps=9)

    # --- Build selection for clustered core residues ---
    selection = 'resindex '
    for item in res_list:
        selection += str(item) + ' or resindex '
    selection = selection[0:-13]

    sel = u.select_atoms(selection)

    # --- Write clustered residues to temp PDB and reload ---
    with mda.Writer('spatial.pdb') as pdb:
        pdb.write(sel)
    univ = mda.Universe('spatial.pdb')

    # --- Run GITIM surface analysis ---
    g = univ.select_atoms('name CA')
    interface = pytim.GITIM(
        univ,
        group=g,
        alpha=3.9,
        max_layers=2,
        molecular=False,
        centered='middle'
    )

    # --- Extract layer 1 (outer) and layer 2 (inner) via interface object ---
    layer1_atoms = interface.layers[0]   # outer surface
    layer2_atoms = interface.layers[1]   # mid layer
    #layer3_atoms = interface.layers[2]   # inner layer

    outer_resids = np.unique(layer1_atoms.resids)
    #inner_resids = np.unique(layer2_atoms.resids)
     
    inner_resids = np.setdiff1d(res_list, outer_resids)

    print(len(outer_resids), len(inner_resids), len(res_list))

    return inner_resids, outer_resids, res_list



# define functions
from scipy import optimize
from scipy import sparse

def timeCorr_bool(mats, N, m):
    # mats is a list of bool matrix, which express the bindings
    # N= len(mats)
    # m: time region of correlation, should < N/2
    corr_m= np.zeros(m)
    #cnt= 1
    for i in range(m):
        # matCorr= []
        matCorr= np.zeros(N-i)
        for k in range(N-i):
            matCorr[k]= (mats[k] & mats[k+i]).sum()
            # matCorr.append( (mats[k] * mats[k+i]).sum() )
        corr_m[i]= np.mean(matCorr)
        
        '''if i>= ((cnt*m)//25)-1:
            print(f"{cnt*25} %")
            cnt+=1'''
        
    corrN_m= corr_m/corr_m[0]
    return corrN_m, corr_m

def timeCorr_boolCsr(mats, N, m):
    # mats is a list of bool csr_matrix, which express the bindings
    # N= len(mats)
    # m: time region of correlation, should < N/2
    corr_m= np.zeros(m)
    # cnt= 1
    for i in range(m):
        # matCorr= []
        matCorr= np.zeros(N-i)
        for k in range(N-i):
            matCorr[k]= (mats[k].multiply(mats[k+i])).sum()
        corr_m[i]= np.mean(matCorr)
        
        '''if i>= ((cnt*m)//10)-1:
            print(f"{cnt*10} %")
            cnt+=1'''
        
    corrN_m= corr_m/corr_m[0]
    return corrN_m, corr_m

def exp2(t, a, tau1, c):
    return a * np.exp(-t / tau1) + (1 - a) * np.exp(-t / (c*tau1))

def fit_corr(Ct, t, init):
    # fit normalized correlation ( C(0)=1 )
    n= len(t)
    popt2, pcov2 = optimize.curve_fit(
            exp2, t, Ct, sigma= (t+t[1])**(1/3.0),
            bounds=([0,0,1],[1,10*t[-1],math.inf]), p0=init, maxfev=1000000) # sigma= t**0.5,
    
    Ct_fit= np.zeros(n)
    for i in range(n):
        Ct_fit[i]= exp2(t[i], *popt2)
    return Ct_fit, popt2    

def calc_contact(grp1, grp2, d, dims):
    # calculate contact matrix
    dist= contacts.distance_array(grp1.positions, grp2.positions, box=dims)
    contMat_12= contacts.contact_matrix(dist, radius=d)
    return contMat_12


def timeCorr_boolCsr_fromStart(mats, N, m):
    # mats is a list of bool csr_matrix, which express the bindings
    # N= len(mats)
    # m: time region of correlation, should < N/2
    corr_m = np.zeros(m)
    
    for i in range(m):
        corr_m[i] = (mats[0].multiply(mats[i])).sum()
    
    corrN_m = corr_m / corr_m[0]
    return corrN_m, corr_m


def visul5(unive,s1,s2):
    view=nv.show_mdanalysis(unive)
    view.clear_representations()
    view.add_representation('ball+stick',sele=f"(nucleic) and ( {s1})",
                            opacity=1.5,aspectRatio=2,color='red')  
    
    view.add_representation('ball+stick',sele=f"(protein) and ( {s1})",
                            opacity=1.5,aspectRatio=2,color='yellow')  
    

    view.add_representation('ball+stick',sele=f"(nucleic or protein ) and ({s2})",
                            opacity=1.5,
                            aspectRatio=2,
                            color='green')
    
    

    view.add_unitcell(color='white')
    return view


In [ ]:

# ══════════════════════════════════════════════════════════════════════════════
#  Main analysis loop
# ══════════════════════════════════════════════════════════════════════════════

systems    = ['2_close', 
             '2_spread', '3_close', '3_spread'
             ]
clrs       = ['#E01010', '#1010E0', '#10A010', '#A010A0', '#10A0A0']
clrs2      = ['#E09010', '#1090E0', '#10C040', '#C040C0', '#40C0C0']

# ── Parameters ────────────────────────────────────────────────────────────────
d_cont     = 4.3        # Å  contact cutoff
chain_size = 9        # residues per RNA chain
star       = 5000     # frames to analyse
m_corr     = 2000     # correlation window  (~2830)
dt         = 2e-3     # ps per frame → ns  (2 ps = 0.002 ns)
eps_clust  = 9        # DBSCAN eps for clustering

offsets    = [0]   # multiple trajectory start offsets

save_dir   = './corr_results'
os.makedirs(save_dir, exist_ok=True)

results = {}
all_contacts = {}

for idx, label in enumerate(systems):

    print(f"\n{'='*60}")
    print(f"  Processing: {label}")
    print(f"{'='*60}")

    # ── Load universe ─────────────────────────────────────────────────────────
    stru = parmed.load_file(f'../gxg_comps/9_res/equal/{label}/output.top')
    stru.strip('@OW,HW1,HW2')
    u = mda.Universe(stru, f'../gxg_comps/9_res/equal/{label}/traj-nw.dcd')

    a        = u.atoms
    workflow = [trans.wrap(a), trans.unwrap(a)]
    u.trajectory.add_transformations(*workflow)

    # ── Time axis (in ns) — shared across all offsets ─────────────────────────
    tM = dt * np.arange(m_corr)   # ns

    # ── Multi-offset loop: clustering + contacts + correlation ────────────────
    all_corrN  = []   # normalised correlations, one per offset
    all_corr   = []   # raw correlations, one per offset

    for number in offsets:
        traj_start = -(star + number)
        print(f"\n  --- Offset {number} (traj slice [{traj_start}:]) ---")

        # ── Clustering at the start frame of this window ───────────────────
        ts = u.trajectory[traj_start]
        centroid, to_cluster, clustered_list, res_list = clustering(
            u, ts, eps=eps_clust
        )

        residue_list = ' '
        for item in res_list:
            residue_list += str(item) + ' '

        # ── Atom selections (re-derived per offset) ────────────────────────
#         idp_1 = u.select_atoms(f"name CA and (resid {residue_list})")[::2]
#         idp_2 = u.select_atoms(f"name CA and (resid {residue_list})")

        idp_1 = u.select_atoms(f"(protein and (not name H*)) and (resid {residue_list})")[::]
        idp_2 = u.select_atoms(f"(protein and (not name H*)) and (resid {residue_list})")        

        # ── Intra-chain mask ───────────────────────────────────────────────
        resids_1   = idp_1.resids
        resids_2   = idp_2.resids
        chain_1    = resids_1 // chain_size
        chain_2    = resids_2 // chain_size
        
        # same-chain mask
        same_chain = (chain_1[:, None] == chain_2[None, :])
        
        # sequence separation
        seq_sep = np.abs(resids_1[:, None] - resids_2[None, :])
        
        exclude_mask = same_chain & (seq_sep <= 3)

        print(f"    idp_1: {len(idp_1)} atoms  |  idp_2: {len(idp_2)} atoms  |  "
              f"chains: {chain_1.max() + 1}  |  dilute residues: {len(res_list)}")

        # ── Contact matrix accumulation ────────────────────────────────────
        contMt = []
        for _ in u.trajectory[traj_start:traj_start+5]:
            contMat = calc_contact(idp_1, idp_2, d_cont, u.dimensions)
            contMat[exclude_mask] = False
            contMt.append(contMat.sum()/len(contMat.sum(axis=1)))
            #contMt.append(sparse.csr_matrix(contMat))
            
        # convert to array
        contMt_array = np.array(contMt).flatten()

        # store in dictionary
        all_contacts[label] = contMt_array

df_contMt = pd.DataFrame(all_contacts)

# ============================================================
# SAVE CSV
# ============================================================

df_contMt.to_csv("contact_matrix_summary_all_systems.csv",
                 index=False)


In [ ]:
df_contMt.plot()

In [ ]:
df_contMt

In [ ]:
u = mda.Universe('frame2_104679.pdb')
u.select_atoms('(protein and (not name H*)) and (resid 1 )')

In [ ]:

# ══════════════════════════════════════════════════════════════════════════════
#  Main analysis loop
# ══════════════════════════════════════════════════════════════════════════════

systems    = ['2_close', 
             '2_spread', '3_close', '3_spread'
             ]
clrs       = ['#E01010', '#1010E0', '#10A010', '#A010A0', '#10A0A0']
clrs2      = ['#E09010', '#1090E0', '#10C040', '#C040C0', '#40C0C0']

# ── Parameters ────────────────────────────────────────────────────────────────
d_cont     = 4.0        # Å  contact cutoff
chain_size = 9        # residues per RNA chain
star       = 5000     # frames to analyse
m_corr     = 2000     # correlation window  (~2830)
dt         = 2e-3     # ps per frame → ns  (2 ps = 0.002 ns)
eps_clust  = 9        # DBSCAN eps for clustering

offsets    = [0]   # multiple trajectory start offsets

save_dir   = './corr_results'
os.makedirs(save_dir, exist_ok=True)

results = {}
all_contacts = {}

for idx, label in enumerate(systems):

    print(f"\n{'='*60}")
    print(f"  Processing: {label}")
    print(f"{'='*60}")

    # ── Load universe ─────────────────────────────────────────────────────────
    stru = parmed.load_file(f'../gxg_comps/9_res/equal/{label}/output.top')
    stru.strip('@OW,HW1,HW2')
    u = mda.Universe(stru, f'../gxg_comps/9_res/equal/{label}/traj-nw.dcd')

    a        = u.atoms
    workflow = [trans.wrap(a), trans.unwrap(a)]
    u.trajectory.add_transformations(*workflow)

    # ── Time axis (in ns) — shared across all offsets ─────────────────────────
    tM = dt * np.arange(m_corr)   # ns

    # ── Multi-offset loop: clustering + contacts + correlation ────────────────
    all_corrN  = []   # normalised correlations, one per offset
    all_corr   = []   # raw correlations, one per offset

    for number in offsets:
        traj_start = -(star + number)
        print(f"\n  --- Offset {number} (traj slice [{traj_start}:]) ---")

        # ── Clustering at the start frame of this window ───────────────────
        ts = u.trajectory[traj_start]
        centroid, to_cluster, clustered_list, res_list = clustering(
            u, ts, eps=eps_clust
        )

        residue_list = ' '
        for item in res_list:
            residue_list += str(item) + ' '

        # ── Atom selections (re-derived per offset) ────────────────────────
#         idp_1 = u.select_atoms(f"name CA and (resid {residue_list})")[::2]
#         idp_2 = u.select_atoms(f"name CA and (resid {residue_list})")

        idp_1 = u.select_atoms(f"(protein and (not name H*)) and (resid {residue_list})")[::]
        idp_2 = u.select_atoms(f"(protein and (not name H*)) and (resid {residue_list})")        

        # ── Intra-chain mask ───────────────────────────────────────────────
        resids_1   = idp_1.resids
        resids_2   = idp_2.resids
        chain_1    = resids_1 // chain_size
        chain_2    = resids_2 // chain_size
        intra_mask = (chain_1[:, None] == chain_2[None, :])

        print(f"    idp_1: {len(idp_1)} atoms  |  idp_2: {len(idp_2)} atoms  |  "
              f"chains: {chain_1.max() + 1}  |  dilute residues: {len(res_list)}")

        # ── Contact matrix accumulation ────────────────────────────────────
        contMt = []
        for _ in u.trajectory[traj_start:traj_start+5]:
            contMat = calc_contact(idp_1, idp_2, d_cont, u.dimensions)
            contMat[intra_mask] = False
            contMt.append(contMat.sum()/len(contMat.sum(axis=1)))
            #contMt.append(sparse.csr_matrix(contMat))
            
        # convert to array
        contMt_array = np.array(contMt).flatten()

        # store in dictionary
        all_contacts[label] = contMt_array

df_contMt = pd.DataFrame(all_contacts)

# ============================================================
# SAVE CSV
# ============================================================

df_contMt.to_csv("contact_matrix_summary_all_systems.csv",
                 index=False)


In [ ]:
df_contMt

In [ ]:
df_contMt.plot()

In [ ]:
contMat.sum()/len(contMat.sum(axis=1))

In [ ]:
len(contMat.sum(axis=1))

In [ ]:
intra_mask

In [ ]:
resids_2[:-10]// chain_size, resids_2[:-10]

In [ ]:
direct_conts = pd.read_csv(f'/Volumes/potoyan-lab/will/gxg-project/gxg_comps/3_9_conts_timeseries.csv', 
                           index_col=0, low_memory=False)
direct_conts

In [ ]:
close_spread = pd.read_csv(f'/Volumes/potoyan-lab/will/gxg-project/gxg_comps/close_spread_conts_timeseries.csv', 
                           index_col=0, low_memory=False)
close_spread

In [ ]:
direct_conts = pd.concat([direct_conts, close_spread], axis=1)

# direct_conts = direct_conts.rename(columns={'3_E':'3E', '3_C':'3C', '3_A':'3A', '3_P':'3P',
#        '3_N':'3N', '9_E':'9E', '9_C':'9C', '9_A':'9A',
#        '9_P':'9P', '9_N':'9N', '2_close':'GGGXGXGGG','2_spread':'GXGGGGGXG', '3_close':'GGGXXXGGG',
#        '3_spread':'GXGGXGGXG'})



direct_conts = direct_conts.rename(columns={'3_E':'3E', '3_C':'3C', '3_A':'3A', '3_P':'3P',
       '3_N':'3N', '9_E':'9E', '9_C':'9C', '9_A':'9A',
       '9_P':'9P', '9_N':'9N', '2_close':'X2','2_spread':'X2S', '3_close':'X3',
       '3_spread':'X3S'})


direct_conts

In [ ]:
n_res_dict = {'3E': 900,
 '3C': 870,
 '3A': 909,
 '3P': 870,
 '3N': 876,
 '9E': 900,
 '9C': 855,
 '9A': 756,
 '9P': 855,
 '9N': 738,
 'X2': 720,
 'X2S': 720,
 'X3': 720,
 'X3S': 720}

In [ ]:
all_conts = pd.DataFrame()
for i,j in n_res_dict.items():
    for k in direct_conts.columns:
        if i == k:
            all_conts.loc['Direct',f'{k}'] = direct_conts[k].mean() / n_res_dict[k]

In [ ]:
all_conts

In [ ]:
systems = all_conts.columns
values = all_conts.loc['Direct']

# ============================================================
# COLOR MAP
# ============================================================

color_map = {
    '3E': '#D62728',
    '3C': '#1F77B4',
    '3A': '#9467BD',
    '3P': '#2CA02C',
    '3N': '#FF7F0E',

    '9E': '#D62728',
    '9C': '#1F77B4',
    '9A': '#9467BD',
    '9P': '#2CA02C',
    '9N': '#FF7F0E',

    'X2' : '#8C564B',
    'X2S': '#E377C2',
    'X3' : '#7F7F7F',
    'X3S': '#BCBD22'
}

colors = [color_map[s] for s in systems]

# ============================================================
# PLOT
# ============================================================

fig, ax = plt.subplots(figsize=(8,3))

ax.bar(systems,
       values,
       color=colors)

# ============================================================
# GROUP SEPARATORS
# ============================================================

ax.axvline(4.5, linestyle='--', linewidth=1)
ax.axvline(9.5, linestyle='--', linewidth=1)

# ============================================================
# LABELS
# ============================================================

ax.set_ylabel('Direct Contacts')
ax.set_xlabel('Systems')
plt.ylim(6,8)
plt.tight_layout()
plt.show()

In [ ]:
x = pd.read_csv('../gxg_comps/3_res/charged/all_size.csv')
x.tail()

In [ ]:
x['ALA'][1]

In [ ]:
from MDAnalysis.analysis import distances
from sklearn.cluster import DBSCAN
from collections import Counter
import MDAnalysis as mda
from parmed import load_file
from MDAnalysis.analysis.hydrogenbonds import HydrogenBondAnalysis

def clustering(u, frame, dens):

    ### Function that is nested in all of the analysis methods in this file ###

    ca_sel = u.select_atoms('resid 2 and name CA')

    cas = [atom.index for atom in ca_sel.atoms]

    to_cluster = [atom for i, atom in enumerate(u.trajectory[frame].positions)if i in cas]

    n_ca = len(ca_sel)

    self_distances = distances.self_distance_array(ca_sel.positions,
                                                   box=u.trajectory[frame].dimensions)

    sq_dist_arr = np.zeros((n_ca, n_ca))
    triu = np.triu_indices_from(sq_dist_arr, k=1)

    sq_dist_arr[triu] = self_distances
    sq_dist_arr.T[triu] = self_distances

    if dens == 50:
        model = DBSCAN(eps=10, min_samples=1, metric='precomputed')
    elif dens == 125:
        model = DBSCAN(eps=9, min_samples=2, metric='precomputed')
    else:
        model = DBSCAN(eps=10, min_samples=2, metric='precomputed')

    labels = model.fit_predict(sq_dist_arr)
    b = Counter(labels)

    clustered_list = [atom for j, atom in enumerate(to_cluster) 
                      for i, item in enumerate(labels) 
                      if item == b.most_common()[0][0] 
                      and j == i]

    centroid = np.mean(clustered_list, axis=0)

    ind_list = [cas[index] for index, j in enumerate(to_cluster) 
                for i in clustered_list 
                if j[0]==i[0] and j[1]==i[1] and j[2]==i[2]]

    index_list = [x+1 for x in ind_list]

    res_list = [u.atoms[i].resindex for i in index_list]
    
    name_list = [u.atoms[i].resname for i in index_list]


    return centroid, to_cluster, clustered_list, res_list, name_list

In [ ]:
from collections import Counter, defaultdict
import pandas as pd

# Initialize dictionary to store counts per residue
# Each key = residue name, value = list of counts over frames

def aa_number(u,start=0,stop=-1,step=1):

    residue_counts = defaultdict(list)

    for ts in u.trajectory[start:stop:step]:
        centroid, to_cluster, clustered_list, res_list, name_list = clustering(
            u, frame=ts.frame, dens=50
        )

        counts = Counter(name_list)

        # Update dict for each residue seen so far
        for res in counts:
            residue_counts[res].append(counts[res])

        # Ensure residues not present in this frame get a 0
        for res in residue_counts:
            if res not in counts:
                residue_counts[res].append(0)


    df = pd.DataFrame(residue_counts)

    # Initialize dictionary of residue totals
    residue_totals = {}

    # Loop over all columns in df (each amino acid)
    for res in df.columns:
        n_res = len(u.select_atoms(f"resname {res} and name CA and (resid 2)").resids)
        residue_totals[res] = n_res

    # Convert to series for alignment
    residue_totals = pd.Series(residue_totals)

    # Normalize df by dividing each column by its residue count
    df_normalized = df / residue_totals
    
    return df_normalized



In [ ]:
#stru = parmed.load_file('../gxg_comps/3_res/equal/output.top')
stru ='../gxg_comps/3_res/polar//output-nw.gro'
# or explicitly by atom names:
#stru.strip('@OW,HW1,HW2')

u = mda.Universe(stru,
                 '../gxg_comps/3_res/polar//traj_2-nw.dcd')

In [ ]:
df_polar = aa_number(u,start=20000,step=20010)

In [ ]:
import matplotlib.pyplot as plt

# Compute mean and std
means = df_polar.mean(axis=0)
stds  = df_polar.std(axis=0)

# Sort by mean values
means_sorted = means.sort_values()
stds_sorted = stds[means_sorted.index]  # align stds with new order

# Plot
plt.figure(figsize=(10, 5))
plt.bar(means_sorted.index, means_sorted.values,
        yerr=stds_sorted.values, capsize=4,
        alpha=0.7, color="skyblue")

plt.ylabel("Fraction (avg ± std)")
plt.xlabel("Amino Acid")
plt.title("Average Fraction of Each Amino Acid (Sorted Ascending)")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()



In [ ]:
df_polar.to_csv('', index=False)

#### RDF around COMS

In [ ]:
import MDAnalysis as mda
import numpy as np
import pandas as pd

stru = parmed.load_file('../gxg_comps/3_res/polar/output.top')
stru.strip('@OW,HW1,HW2')
# Load Universe
u = mda.Universe(
    stru,'../gxg_comps/3_res/polar/traj_2-nw.dcd'
)


In [ ]:
# Load Universe
stru = parmed.load_file('../gxg_comps/9_res/polar//output.top',
                        xyz='../gxg_comps/9_res/polar//output.gro')
stru.strip('@OW,HW1,HW2')
# Load Universe
u = mda.Universe(
    stru,'../gxg_comps/9_res/polar//traj-nw.dcd'
)


In [ ]:
u.select_atoms('resname ARG and name CA').indices

In [ ]:

# List of amino acids in your system
amino_acids = sorted(set(u.select_atoms("protein").resnames))
amino_acids=['LYS','ILE','ALA','ARG','GLU']

In [ ]:
import numpy as np
import pandas as pd

# Define bins for RDF
bins = np.linspace(0,50,30)   # 0–100 Å, 20 bins
bin_centers = 0.5 * (bins[1:] + bins[:-1])

# Shell volumes (Å^3)
shell_volumes = (4/3) * np.pi * (bins[1:]**3 - bins[:-1]**3)

# Storage for results
rdf_data = {res: np.zeros(len(bins) - 1) for res in amino_acids}

nframes = 1
# Loop over trajectory
for ts in u.trajectory[50000:-1:100]:
    
    centroid, to_cluster, clustered_list, res_list, name_list = clustering(
    u, frame=ts.frame, dens=50
        )
    
    # Center of mass of entire protein
    #com = u.select_atoms("protein").center_of_mass()
    com = centroid
    # Loop over amino acid types

    for res in amino_acids:
        sel = u.select_atoms(f"resname {res}  and not (name H* )")
        if len(sel) == 0:
            continue

        # Residue COMs
        res_coms = [r.atoms.center_of_mass() for r in sel.residues]
        dists = np.linalg.norm(res_coms - com, axis=1)

        # Histogram distances
        hist, _ = np.histogram(dists, bins=bins)
        rdf_data[res] += hist
        
        
    nframes += 1

# Convert counts → density (per Å^3 per frame)
rdf_density = {res: rdf_data[res] / (shell_volumes * nframes) for res in amino_acids}

# Store in DataFrame
rdf_df = pd.DataFrame(rdf_density, index=bin_centers)

# Save to CSV if needed
#rdf_df.to_csv("aminoacid_rdf.csv")


In [ ]:
len(dists)

In [ ]:
### amino_acids=['ILE','ALA','ARG','LYS']
for i in amino_acids:
    plt.plot(rdf_df.index,rdf_df[i],label=i)


plt.legend()

In [ ]:
res_coms

In [ ]:
plt.plot(rdf_df['ARG'])

In [ ]:
import MDAnalysis as mda
import numpy as np
import matplotlib.pyplot as plt

def rdf_of_droplet(u,start=0,stop=-1,select='name OW'):
    
    #droplet_atoms = u.select_atoms("all")
    interest = u.select_atoms(select)
    
    # === Parameters for RDF calculation ===
    r_max = 1000.0  # max distance for RDF (Å)
    bin_width = 5.0
    bins = np.arange(0, r_max + bin_width, bin_width)
    rdf_counts = np.zeros(len(bins) - 1)

    # System density (needed for normalization)
    volume = np.prod(u.dimensions[:3])  # Å^3
    #number_density = len(droplet_atoms) / volume
    number_density = len(interest) / volume
    # === Loop over frames ===
    i=1
    for ts in u.trajectory[start:stop]:
        # Droplet center of mass
        droplet_atoms = u.select_atoms("Protein")
        
        centroid, to_cluster, clustered_list, res_list, name_list = clustering(
    u, frame=ts.frame, dens=50
        )
        
        interest = u.select_atoms(select)

        # Distances of all atoms from COM
        distances = np.linalg.norm(interest.positions - centroid, axis=1)

        # Histogram the distances
        counts, _ = np.histogram(distances, bins=bins)
        rdf_counts += counts
        
        i+=1

        # === Normalize RDF ===
    # Average over number of frames
    rdf_counts /= i

    # Normalization by ideal gas distribution
    r = 0.5 * (bins[1:] + bins[:-1])  # bin centers
    shell_volumes = 4.0 / 3.0 * np.pi * (bins[1:]**3 - bins[:-1]**3)
    ideal_counts = number_density * shell_volumes
    rdf = rdf_counts / ideal_counts
    
    return r,rdf


In [ ]:
u.select_atoms('resname LYS').indices

In [ ]:
df_normalized

In [ ]:
residue_totals

In [ ]:
from collections import Counter

counts = Counter(name_list)
print(counts) 


In [ ]:
u.select_atoms('resname ASN and (name CA)')
#u.select_atoms('resname LEU and (name CA)')

In [ ]:
u.select_atoms('resname ARG and (name CA)')